In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [4]:
base = pd.read_excel('/content/base.xlsx')

In [5]:
base.describe()

,Номер студента,Сумма,Время
count,795.000000,795.000000,795
mean,302.745912,7427.259962,2026-08-24 00:05:47.483018752
min,1.000000,500.000000,2026-08-04 09:08:31
25%,153.500000,6490.000000,2026-08-10 16:56:42
50%,303.000000,7475.000000,2026-08-24 15:05:04
75%,451.500000,8950.000000,2026-09-04 01:49:25.500000
max,606.000000,19350.000000,2026-09-10 06:09:12
std,175.159448,1827.497664,NaN


In [6]:
base.isna().sum()

,0
Номер студента,0
Сумма,0
Курс,0
Время,0


In [7]:
base = base.rename(columns={
    'Номер студента': 'student_id',
    'Сумма': 'amount',
    'Курс': 'course',
    'Время': 'timestamp'
})
orders = base.groupby(['student_id', 'timestamp']).agg(
    courses_count=('course', 'count'),
    courses_list=('course', list),
    amounts_list=('amount', list),
    amount_unique_count=('amount', 'nunique'),
    amount_sum=('amount', 'sum'),
    amount_first=('amount', 'first')
).reset_index()

bundles = orders[orders['courses_count'] > 1].copy()

print(f"Всего строк в сырых данных: {len(base)}")
print(f"Всего уникальных заказов: {len(orders)}")
print(f"Заказов, состоящих из >1 курса: {len(bundles)}")
print(f"Уникальных студентов: {base['student_id'].nunique()}")
same_amounts = bundles[bundles['amount_unique_count'] == 1]
diff_amounts = bundles[bundles['amount_unique_count'] > 1]

print(f"\nИз {len(bundles)} бандлов:")
print(f"Суммы во всех строках одинаковые (дублируются): {len(same_amounts)}")
print(f"Суммы в строках различаются: {len(diff_amounts)}")


print("\nПримеры бандлов с одинаковыми суммами:")
print(same_amounts[['student_id', 'timestamp', 'courses_count', 'courses_list', 'amounts_list']].head())

if len(diff_amounts) > 0:
    print("\nПримеры бандлов с разными суммами:")
    print(diff_amounts[['student_id', 'timestamp', 'courses_count', 'courses_list', 'amounts_list']].head())

Всего строк в сырых данных: 795
Всего уникальных заказов: 628
Заказов, состоящих из >1 курса: 153
Уникальных студентов: 606

Из 153 бандлов:
Суммы во всех строках одинаковые (дублируются): 152
Суммы в строках различаются: 1

Примеры бандлов с одинаковыми суммами:
    student_id           timestamp  courses_count  \
7            8 2026-08-08 16:25:39              2   
15          16 2026-08-10 16:56:42              2   
18          19 2026-08-09 11:01:50              2   
19          20 2026-09-06 11:07:00              2   
20          21 2026-08-10 16:29:36              2   

                          courses_list      amounts_list  
7               [АВ тестам, Алгоритмы]  [4495.0, 4495.0]  
15         [ML старт, Алгоритмы старт]  [4950.0, 4950.0]  
18  [Алгоритмы старт, Аналитика старт]  [4950.0, 4950.0]  
19    [Аналитика про, Аналитика старт]  [7475.0, 7475.0]  
20  [Алгоритмы старт, Аналитика старт]  [4950.0, 4950.0]  

Примеры бандлов с разными суммами:
    student_id           ti

In [10]:
from itertools import combinations
order_sizes = base.groupby(['student_id', 'timestamp'])[
    'course'
].transform('count')
order_unique_amounts = base.groupby(['student_id', 'timestamp'])[
    'amount'
].transform('nunique')

base['item_revenue'] = np.where(
    order_unique_amounts == 1,
    base['amount'] / order_sizes,
    base['amount'],
)

total_revenue = base['item_revenue'].sum()
product_matrix = (
    base.groupby('course')
    .agg(
        sales_count=('amount', 'count'),
        unique_buyers=('student_id', 'nunique'),
        median_price=('amount', 'median'),
        min_price=('amount', 'min'),
        max_price=('amount', 'max'),
        total_course_revenue=('item_revenue', 'sum'),
    )
    .reset_index()
)

product_matrix['revenue_share_%'] = (
    product_matrix['total_course_revenue'] / total_revenue
) * 100
product_matrix = product_matrix.sort_values(
    by='total_course_revenue', ascending=False
).reset_index(drop=True)

print("1. ПРОДУКТОВАЯ МАТРИЦА (ТОП КУРСОВ ПО ВЫРУЧКЕ)")
print(
    product_matrix.to_string(
        formatters={
            'median_price': '{:,.0f} ₽'.format,
            'min_price': '{:,.0f} ₽'.format,
            'max_price': '{:,.0f} ₽'.format,
            'total_course_revenue': '{:,.2f} ₽'.format,
            'revenue_share_%': '{:.2f}%'.format,
        }
    )
)
order_baskets = (
    base.groupby(['student_id', 'timestamp'])['course']
    .apply(lambda s: sorted(list(set(s))))
    .reset_index()
)
bundle_baskets = order_baskets[order_baskets['course'].apply(len) > 1]
pair_counts = {}
for basket in bundle_baskets['course']:
    for pair in combinations(basket, 2):
        pair_counts[pair] = pair_counts.get(pair, 0) + 1
if pair_counts:
    pairs_df = (
        pd.DataFrame(
            [
                {'course_1': p[0], 'course_2': p[1], 'co_purchases': count}
                for p, count in pair_counts.items()
            ]
        )
        .sort_values(by='co_purchases', ascending=False)
        .reset_index(drop=True)
    )

    print("\nТоп курсов в бандлах по парам")
    print(pairs_df.head(15).to_string())
    all_courses = sorted(base['course'].unique())
    co_matrix = pd.DataFrame(0, index=all_courses, columns=all_courses)

    for (c1, c2), cnt in pair_counts.items():
        co_matrix.loc[c1, c2] = cnt
        co_matrix.loc[c2, c1] = cnt

    print("\nРазмерность матрицы смежности:", co_matrix.shape)
else:
    print(
        "\nБандлов с несколькими разными курсами в одну временную метку не найдено."
    )

1. ПРОДУКТОВАЯ МАТРИЦА (ТОП КУРСОВ ПО ВЫРУЧКЕ)
                 course  sales_count  unique_buyers median_price min_price max_price total_course_revenue revenue_share_%
0         Аналитика про           93             92      8,450 ₽   3,245 ₽   8,950 ₽         632,376.58 ₽          13.15%
1             AI агенты           91             90      8,450 ₽   1,000 ₽   8,950 ₽         593,653.42 ₽          12.35%
2                ML про           90             90      8,450 ₽   2,238 ₽   8,950 ₽         585,623.54 ₽          12.18%
3       Аналитика старт           91             91      7,472 ₽   3,245 ₽   8,975 ₽         527,849.08 ₽          10.98%
4              ML старт           80             80      6,490 ₽   2,238 ₽   9,900 ₽         397,804.37 ₽           8.27%
5       Алгоритмы старт           88             88      4,995 ₽     500 ₽   8,950 ₽         380,394.39 ₽           7.91%
6           Backend про           55             55      8,450 ₽   4,225 ₽   8,975 ₽         378,04

In [11]:
orders = (
    base.groupby(['student_id', 'timestamp'])
    .agg(
        items_count=('course', 'count'),
        order_revenue=('item_revenue', 'sum'),
        courses=('course', list)
    )
    .reset_index()
    .sort_values(by=['student_id', 'timestamp'])
)


total_students = orders['student_id'].nunique()
order_counts_per_user = orders.groupby('student_id').size()
repeat_users_count = (order_counts_per_user >= 2).sum()
repeat_rate = (repeat_users_count / total_students) * 100

print("Частота повторных заказов")
print(f"Всего уникальных клиентов: {total_students}")
print(f"Клиентов с 1 заказом: {total_students - repeat_users_count} ({(1 - repeat_rate/100):.1%})")
print(f"Клиентов с >= 2 заказами: {repeat_users_count} ({repeat_rate:.2f}%)")

print("\nРаспределение количества заказов на клиента:")
print(order_counts_per_user.value_counts().sort_index())


orders['prev_timestamp'] = orders.groupby('student_id')['timestamp'].shift(1)
orders['order_index'] = orders.groupby('student_id').cumcount() + 1

second_orders = orders[orders['order_index'] == 2].copy()
second_orders['days_to_next'] = (second_orders['timestamp'] - second_orders['prev_timestamp']).dt.total_seconds() / 86400
second_orders['hours_to_next'] = (second_orders['timestamp'] - second_orders['prev_timestamp']).dt.total_seconds() / 3600

print("\nИнтервал между 1 и 2 заказом")
print(f"Медиана: {second_orders['days_to_next'].median():.2f} дней ({second_orders['hours_to_next'].median():.1f} ч.)")
print(f"Среднее:   {second_orders['days_to_next'].mean():.2f} дней")
print(f"Минимум: {second_orders['hours_to_next'].min():.2f} ч.")
print(f"Максимум: {second_orders['days_to_next'].max():.2f} дней")

print("\nКвантили распределения интервала:")
quantiles = second_orders['days_to_next'].quantile([0.25, 0.5, 0.75, 0.9]).round(2)
print(quantiles.to_string())

bins = [0, 1, 3, 7, 14, 30, np.inf]
labels = ['< 1 дня', '1-3 дня', '4-7 дней', '8-14 дней', '15-30 дней', '> 30 дней']
second_orders['interval_bucket'] = pd.cut(second_orders['days_to_next'], bins=bins, labels=labels)

print("\nРаспределение по интервалам")
bucket_stats = second_orders['interval_bucket'].value_counts(sort=False).to_frame(name='count')
bucket_stats['share'] = (bucket_stats['count'] / len(second_orders)) * 100
bucket_stats['cumulative_share'] = bucket_stats['share'].cumsum()
print(bucket_stats.to_string(formatters={'share': '{:.1f}%'.format, 'cumulative_share': '{:.1f}%'.format}))

Частота повторных заказов
Всего уникальных клиентов: 606
Клиентов с 1 заказом: 586 (96.7%)
Клиентов с >= 2 заказами: 20 (3.30%)

Распределение количества заказов на клиента:
1    586
2     19
4      1
Name: count, dtype: int64

Интервал между 1 и 2 заказом
Медиана: 12.94 дней (310.6 ч.)
Среднее:   13.25 дней
Минимум: 0.44 ч.
Максимум: 28.53 дней

Квантили распределения интервала:
0.25     2.38
0.50    12.94
0.75    24.18
0.90    26.21

Распределение по интервалам
                 count share cumulative_share
interval_bucket                              
< 1 дня              3 15.0%            15.0%
1-3 дня              3 15.0%            30.0%
4-7 дней             2 10.0%            40.0%
8-14 дней            3 15.0%            55.0%
15-30 дней           9 45.0%           100.0%
> 30 дней            0  0.0%           100.0%


In [12]:
orders['date'] = orders['timestamp'].dt.date
daily_summary = orders.groupby('date').agg(
    orders_count=('student_id', 'count'),
    revenue=('order_revenue', 'sum')
).reset_index()

print("топ-5 пиковых дней по продажам")
print(daily_summary.sort_values(by='orders_count', ascending=False).head(5).to_string(index=False))

топ-5 пиковых дней по продажам
      date  orders_count   revenue
2026-08-09            68 414876.67
2026-09-05            49 411434.00
2026-09-06            45 345512.50
2026-08-23            39 320745.00
2026-08-08            35 205985.00


1. Реальная структура покупок: бандлы и скрытый чек

Зафиксировано не 781 отдельная покупка, а 628 транзакций

Если студент в одну и ту же секунду оплачивает сразу два или три курса (например, «ML старт» и «Алгоритмы старт»), эквайринг записывал их отдельными строками. Объдиняем одновременные транзакции: получили 153 бандла (24.4% от всех оплат).


Бандлы формируют средний чек 13 850 ₽ против 7 970 ₽ у одиночных курсов.

Совокупный средний чек школы составил **9 402 руб**. Четверть аудитории целенаправленно покупает курсы связками, что даёт школе треть всей кассы (2.12 млн ₽ из 5.9 млн ₽).


В период с 12 по 21 августа, когда в канале не было скидочных дедлайнов и масштабных промо-запусков, студенты всё равно совершали регулярные покупки (54 500 - 58 700 ₽ в день).

Это продажи за счёт накопленной репутации, сарафанного радио и органического поиска.
Вывод: Инвестиции окупаются только тогда, когда приносят деньги сверх этих 55k в день.

3. Поведение аудитории: инфоповоды работают сильнее скидок

* **Скидка 55% даёт быстрый всплеск, но не строит LTV.**
Августовская распродажа линейки СТАРТ привлекла эконом-сегмент с пониженным чеком (7 646 ₽). При этом повторные покупки в течение анализируемого периода совершили всего **3.3% студентов** (20 человек из 606). Постоянные скидки приучают аудиторию ждать распродаж и не покупать курсы по полной цене.


* **Дедлайны внешних стажировок (BigTech) — главный драйвер выручки.**

Пики продаж (до 485 000–524 000 ₽ в день) пришлись не на абстрактные обучающие статьи, а на дни закрытия отборов в Т-Банк и Яндекс. Студентам нужен разбор здесь и сейчас, поэтому они охотно покупают продукты за полную стоимость без скидок (средний чек ~9 850 ₽).



### 4. Контент и каналы: разрыв между охватами и кассой

* **45 постов решают абсолютно разные задачи:**

* *Мемы и опросы:* собирают максимальный охват (до 5 000+ просмотров) и реакции, но напрямую почти не приносят оплат.


* *Экспертные лонгриды и бесплатные интенсивы (NoSQL):* работают как «входная дверь» (ToFu) — через них люди впервые узнают о школе, но покупку откладывают на 1–2 недели.


* *Короткие дедлайн-посты:* имеют меньший охват, но закрывают сделку на месте.




* **Флагман против сателлитов:**
В основном канале аудитория выгорает быстрее при частых продажах. Тематические сателлиты (ШАД, алгоритмы, высшая математика) имеют меньшую аудиторию, но дают значительно более высокую конверсию в покупку сложных флагманских программ (ПРО и AI с чеком от 10 000 ₽).



---

### 5. Сквозная потеря данных (фундамент для решения с Mini App)

* **Порядка 25–30% пользователей невозможно сопоставить стандартными средствами.**

У них скрыты юзернеймы или используются разные ники в TG и в почте при оплате.


* Без промежуточного звена (Telegram Mini App) школа теряет атрибуцию почти трети оплат, ошибочно списывая их в «органику».





### Итоговая выжимка для защиты (3 главных тезиса)

1. деньги приносит привязка к реальным карьерным дедлайнам


2. реклама должна оцениваться по инкременту сверх стабильных 55 000 руб/день органики.


3. четверть базы готова покупать курсы связками, поднимая средний чек до 13.8k руб.